In [13]:
# Data Collection
# Financial data was collected using the simfin Python package. 
# Five files were exported: the income statement, balance sheet, cash flow statement, share prices, and a list of companies.

import simfin as sf
import pandas as pd

sf.set_api_key("4af0e3b6-d15c-4741-9287-80f8c76169aa")
sf.set_data_dir("../dataset/raw/simfin_data/")

tickers = [
    "WMT", "TGT", "KR", "HD", "INTC",
    "AMZN", "LOW", "DG", "DLTR", "BBY",
    "XOM", "CVX", "COP", "SLB", "PSX",
    "KO", "PEP", "CL", "MDLZ",
    "AON"
]

income  = sf.load_income(variant="annual", market="us")
balance = sf.load_balance(variant="annual", market="us")
cashflow = sf.load_cashflow(variant="annual", market="us")
prices  = sf.load_shareprices(variant="daily", market="us")
companies = sf.load_companies(market="us")
industries = sf.load_industries()

Dataset "us-income-annual" on disk (1 days old).
- Loading from disk ... Done!
Dataset "us-balance-annual" on disk (1 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-annual" on disk (1 days old).
- Loading from disk ... Done!
Dataset "us-shareprices-daily" on disk (1 days old).
- Loading from disk ... Done!
Dataset "us-companies" on disk (1 days old).
- Loading from disk ... Done!
Dataset "industries" on disk (1 days old).
- Loading from disk ... Done!


In [14]:
# Selecting Relevant Data
def pick(df):
    return df[df.index.get_level_values("Ticker").isin(tickers)].reset_index()

income, balance, cashflow, prices = map(pick, [income, balance, cashflow, prices])

# Creating the companies Table
companies = companies.reset_index().merge(industries.reset_index(), on="IndustryId")
companies = companies[companies["Ticker"].isin(tickers)]

In [ ]:
# Check which companies are available
def pick(df):
    if "Ticker" in df.index.names:
        df = df.reset_index()
    return df[df["Ticker"].isin(tickers)].copy()

income, balance, cashflow, prices = map(pick, [income, balance, cashflow, prices])

companies = companies.reset_index().merge(industries.reset_index(), on="IndustryId")
companies = companies[companies["Ticker"].isin(tickers)]

datasets = {
    "income": income,
    "balance": balance,
    "cashflow": cashflow,
    "prices": prices,
    "companies": companies,
}

for name, df in datasets.items():
    found = set(df["Ticker"].dropna().unique())
    missing = sorted(set(tickers) - found)
    print(f"{name}: {len(found & set(tickers))}/{len(tickers)} companies, missing: {missing}")

income: 20/20 companies, missing: []
balance: 20/20 companies, missing: []
cashflow: 20/20 companies, missing: []
prices: 20/20 companies, missing: []
companies: 20/20 companies, missing: []


In [16]:
# Saving the Processed Data
income.to_csv("../dataset/raw/income.csv", index=False)
balance.to_csv("../dataset/raw/balance.csv", index=False)
cashflow.to_csv("../dataset/raw/cashflow.csv", index=False)
prices.to_csv("../dataset/raw/prices.csv", index=False)
companies.to_csv("../dataset/raw/companies.csv", index=False)